# Mycovirus discovery workflow orchestration notebook

This notebook orchestrates the existing Slurm/HPC pipeline in this repository. It is intended to **submit and monitor** the wrapper scripts already provided in `Scripts/`, not replace them.

## What this notebook does
- verifies repository location and key files
- sets and validates `CONFIG`
- checks expected directories and adapters
- runs the wrapper scripts in order
- shows basic Slurm monitoring commands
- lists key output directories

## Important
- Run this notebook on a system with access to your Slurm cluster commands (`sbatch`, `squeue`, `sacct`).
- The pipeline scripts are expected to live in `Scripts/`.
- Review each submission cell before running it.


## 1. Configure repository path
Set `REPO_ROOT` to the clone location of `Mycovirus_discovery_workflows`.

In [ ]:
import os
from pathlib import Path

REPO_ROOT = Path(os.environ.get('REPO_ROOT', Path.cwd())).resolve()
print('REPO_ROOT =', REPO_ROOT)
print('Exists:', REPO_ROOT.exists())

In [ ]:
scripts_dir = REPO_ROOT / 'Scripts'
config_path = REPO_ROOT / 'config' / 'pipeline.env'
readme_path = REPO_ROOT / 'README.md'

print('Scripts dir:', scripts_dir, '| exists =', scripts_dir.exists())
print('Config path:', config_path, '| exists =', config_path.exists())
print('README path:', readme_path, '| exists =', readme_path.exists())

required_scripts = [
    '1_setup.sh',
    '2_pipeline_fastqc.sh',
    '3_pipeline_trim.sh',
    '4_pipeline_bowtie2.sh',
    '5_pipeline_spades.sh',
    '6_pipeline_blastx.sh',
    '7_pipeline_summary_result.sh',
]

missing = [name for name in required_scripts if not (scripts_dir / name).exists()]
if missing:
    print('Missing scripts:', missing)
else:
    print('All expected wrapper scripts found.')

## 2. Export CONFIG for downstream shell cells
Adjust this if your config file is in a different location.

In [ ]:
os.environ['REPO_ROOT'] = str(REPO_ROOT)
os.environ['CONFIG'] = str(config_path)
print('CONFIG =', os.environ['CONFIG'])

## 3. Validate the Slurm environment
This checks whether common Slurm commands are available from the notebook kernel environment.

In [ ]:
import shutil
for cmd in ['sbatch', 'squeue', 'sacct', 'bash']:
    print(f'{cmd}:', shutil.which(cmd))

## 4. Inspect configuration and important paths
This shell cell sources `config/pipeline.env` and reports key variables.

In [ ]:
%%bash
set -euo pipefail
export REPO_ROOT="${REPO_ROOT:-$(pwd)}"
export CONFIG="${CONFIG:?CONFIG is not set}"
echo "REPO_ROOT=$REPO_ROOT"
echo "CONFIG=$CONFIG"
test -f "$CONFIG"
source "$CONFIG"
echo "RAW_DIR=${RAW_DIR:-}"
echo "TRIM_DIR=${TRIM_DIR:-}"
echo "MAPPING_DIR=${MAPPING_DIR:-}"
echo "CONTIGS_DIR=${CONTIGS_DIR:-}"
echo "BLAST_DIR=${BLAST_DIR:-}"
echo "LOG_DIR=${LOG_DIR:-}"
echo "ADAPTER_DIR=${ADAPTER_DIR:-}"
if [[ -n "${ADAPTER_DIR:-}" ]]; then
  ls -lah "$ADAPTER_DIR" || true
fi

## 5. Optional preflight checks
This verifies that the main wrapper scripts are executable or runnable with bash.

In [ ]:
%%bash
set -euo pipefail
cd "${REPO_ROOT:?}/Scripts"
pwd
for f in 1_setup.sh 2_pipeline_fastqc.sh 3_pipeline_trim.sh 4_pipeline_bowtie2.sh 5_pipeline_spades.sh 6_pipeline_blastx.sh 7_pipeline_summary_result.sh; do
  if [[ -f "$f" ]]; then
    echo "FOUND $f"
    ls -l "$f"
  else
    echo "MISSING $f"
  fi
done

## 6. Submit setup step
Run this first to create the expected project layout.

In [ ]:
%%bash
set -euo pipefail
cd "${REPO_ROOT:?}/Scripts"
export CONFIG="${CONFIG:?}"
bash ./1_setup.sh

## 7. Submit FastQC
This submits the FastQC wrapper script to Slurm.

In [ ]:
%%bash
set -euo pipefail
cd "${REPO_ROOT:?}/Scripts"
export CONFIG="${CONFIG:?}"
bash ./2_pipeline_fastqc.sh

## 8. Submit trimming
This submits the Trimmomatic wrapper.

In [ ]:
%%bash
set -euo pipefail
cd "${REPO_ROOT:?}/Scripts"
export CONFIG="${CONFIG:?}"
bash ./3_pipeline_trim.sh

## 9. Submit Bowtie2 host removal


In [ ]:
%%bash
set -euo pipefail
cd "${REPO_ROOT:?}/Scripts"
export CONFIG="${CONFIG:?}"
bash ./4_pipeline_bowtie2.sh

## 10. Submit SPAdes / rnaviralspades


In [ ]:
%%bash
set -euo pipefail
cd "${REPO_ROOT:?}/Scripts"
export CONFIG="${CONFIG:?}"
bash ./5_pipeline_spades.sh

## 11. Submit BLASTx search


In [ ]:
%%bash
set -euo pipefail
cd "${REPO_ROOT:?}/Scripts"
export CONFIG="${CONFIG:?}"
bash ./6_pipeline_blastx.sh

## 12. Submit summary step


In [ ]:
%%bash
set -euo pipefail
cd "${REPO_ROOT:?}/Scripts"
export CONFIG="${CONFIG:?}"
bash ./7_pipeline_summary_result.sh

## 13. Optional: run the combined wrapper
Only use this if your repository's `0_Run_all.sh` is already configured for your environment.

In [ ]:
%%bash
set -euo pipefail
cd "${REPO_ROOT:?}/Scripts"
export CONFIG="${CONFIG:?}"
if [[ -f ./0_Run_all.sh ]]; then
  bash ./0_Run_all.sh
else
  echo '0_Run_all.sh not found; skipping.'
fi

## 14. Monitor Slurm jobs
Run these cells after submission to inspect queued/running/completed jobs.

In [ ]:
%%bash
set -euo pipefail
echo "USER=${USER:-$(whoami)}"
squeue -u "${USER:-$(whoami)}" || true

In [ ]:
%%bash
set -euo pipefail
echo 'Recent accounting entries:'
sacct -u "${USER:-$(whoami)}" --format=JobID,JobName%30,State,Elapsed,MaxRSS,ExitCode | tail -n 30 || true

## 15. Inspect outputs
These cells help you quickly see what the pipeline has produced.

In [ ]:
%%bash
set -euo pipefail
export CONFIG="${CONFIG:?}"
source "$CONFIG"
for d in "${FASTQC_DIR:-}" "${TRIM_DIR:-}" "${MAPPING_DIR:-}" "${CONTIGS_DIR:-}" "${BLAST_DIR:-}" "${OUTDIR:-}"; do
  if [[ -n "$d" ]]; then
    echo
    echo "=== $d ==="
    if [[ -d "$d" ]]; then
      find "$d" -maxdepth 2 | head -n 40
    else
      echo 'Directory does not exist yet.'
    fi
  fi
done

## 16. Notes
- Prefer running wrapper scripts with `bash`, not `sh`.
- Review job outputs in `$LOG_DIR` and any per-step log directories.
- For large runs, submit one step and confirm outputs before proceeding to the next.